[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_87_Grounding_Citations_and_Abstention.ipynb)

# Lesson 87 — Grounding, Citations & Abstention
### Phase 10 · RAG at Production Scale · Lesson 5 of ~7

Reranking (L86) got the right chunk to **rank #1**. But retrieval never *guarantees* the
answer is in the context at all — sometimes the corpus simply doesn't contain it. A naive
generator will happily answer anyway, inventing a confident, well-written, **wrong** answer
with no way for the reader to check it. That is the most dangerous RAG failure, because it is
invisible: the output *looks* correct.

This lesson installs the three habits that turn a RAG pipeline into something you can trust in
production:

1. **Grounding** — the answer may use *only* the retrieved context, never the model's own memory.
2. **Citations** — every claim carries a pointer to the exact chunk it came from, so a human (or an eval) can verify it.
3. **Abstention** — when the retrieved context doesn't contain the answer, the correct output is *"I don't know,"* not a guess.

**The failure mode we attack — FM5:** *the model answers confidently from irrelevant or absent context (hallucination), with no provenance.*

## Phase 10 roadmap — the RAG track

| # | Lesson | Failure mode it attacks | Status |
|---|--------|-------------------------|--------|
| L83 | RAG production baseline — where naive RAG breaks | pipeline / retrieval framing | ✅ |
| L84 | Chunking strategies — the ceiling is set at index time | FM4: chunk size & boundaries | ✅ |
| L85 | Dense & hybrid retrieval — embeddings + BM25 + RRF | FM2: the semantic gap | ✅ |
| L86 | Cross-encoder reranking — reorder the top-N precisely | FM3: right chunk retrieved but ranked too low | ✅ |
| **L87** | **Grounding, citations & abstention — answer only from evidence, or refuse** | **FM5: hallucination / no provenance** | **← you are here** |
| L88 | RAG evaluation — retrieval + faithfulness metrics | "it feels better" with no numbers | ⏭ next |
| L89 | Capstone — ship a production RAG service | all of the above, packaged | ⏭ |

Notice the arc: L83–L86 all worked to get **better chunks into the prompt**. L87 is the first
lesson about **what the generator is allowed to do with them**. Even a perfect retriever hands the
model an empty or off-topic context sometimes — this lesson is the safety net for that case.

## §0 — Why "grounded" is a different job from "correct"

A subtle but load-bearing distinction, because it changes what you optimize:

- **Correctness** asks: *is the answer true in the real world?*
- **Faithfulness (grounding)** asks: *is every claim in the answer supported by the retrieved context I was given?*

A RAG system's job is **faithfulness**, not correctness. You cannot make the model omniscient, but
you *can* force it to only assert what its evidence supports — and to admit when the evidence is
missing. This is the contract:

> **Ground truth for a RAG generator is the retrieved context, not the world.**
> If the context is wrong, a faithful answer is wrong *and cites its source* — which is
> debuggable. An unfaithful answer is wrong *and invented* — which is not.

Three mechanisms enforce that contract, and we build all three:

1. **A relevance gate** decides *before generating* whether the context is even good enough to answer. Bad context → abstain.
2. **A grounding instruction** (for an LLM) or **extractive construction** (offline) forces every sentence to come from a chunk.
3. **Citations + a faithfulness self-check** attach and then *verify* the provenance of each claim.

We'll build an offline, deterministic version of all three so the whole notebook runs with **no API
key**, then show the exact Claude prompt that does the same thing with a real model.

## Setup

Fully offline stack: `scikit-learn` for TF-IDF retrieval (same mechanics as L83, dense-swappable),
plus a deterministic extractive generator so every cell runs with **zero API calls**. The real-Claude
grounded generator appears near the end, guarded so it *only* fires if you set an API key — otherwise
it prints a skip note and the notebook still completes clean.

> **Colab:** `Runtime → Run all`. To try the live Claude cell, add `ANTHROPIC_API_KEY` in
> **🔑 Secrets** (left sidebar) and re-run — everything else is identical offline.

In [ ]:
# One-time install (Colab). Offline sandbox already has these.
!pip install scikit-learn anthropic -q

import os, re, textwrap
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Optional API key (Colab Secrets). Absent => notebook runs fully offline.
try:
    from google.colab import userdata          # type: ignore
    if userdata.get("ANTHROPIC_API_KEY"):
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Anthropic API key detected:", HAS_KEY, "(offline is fine — every core cell runs without it)")

## §1 — A small knowledge base

Our corpus is the docs for a fictional product, **Northwind Cloud**. Small enough to read, but it
has the property that matters: it covers *some* topics and not others. That gap is the whole point —
questions inside the docs should be **answered + cited**; questions outside them must be **refused**.

Each chunk is one self-contained fact (L84 taught us why granularity matters). We keep an `id` on
every chunk so a citation can point at it precisely.

In [ ]:
CORPUS = [
    ("D1", "The Northwind Cloud Free tier includes 5 GB of object storage and 10 GB of monthly egress. "
           "Projects on the Free tier are paused after 30 days of inactivity."),
    ("D2", "The Pro tier costs 25 dollars per project per month and raises object storage to 500 GB "
           "with 250 GB of egress included; additional egress is billed at 9 cents per GB."),
    ("D3", "To rotate an API key, open Settings, then API Keys, and click Rotate. The old key keeps "
           "working for a 24 hour grace period so running services do not break during a rotation."),
    ("D4", "Northwind Cloud is available in four regions: us-east, us-west, eu-central, and ap-south. "
           "A project's region is chosen at creation time and cannot be changed afterward."),
    ("D5", "Automated backups run every 6 hours on Pro and Enterprise tiers and are retained for 7 days. "
           "The Free tier does not include automated backups."),
    ("D6", "Team roles are Owner, Admin, and Member. Only an Owner can delete a project or change the "
           "billing plan; an Admin can invite users and manage API keys but cannot delete the project."),
    ("D7", "The command line tool is installed with 'npm install -g northwind-cli' and authenticates "
           "with 'northwind login', which opens a browser to complete the OAuth flow."),
    ("D8", "Rate limits on the Pro tier are 1000 API requests per minute per project. Requests over the "
           "limit receive an HTTP 429 response with a Retry-After header indicating when to try again."),
]
CHUNK_IDS  = [cid for cid, _ in CORPUS]
CHUNK_TEXT = [txt for _, txt in CORPUS]
print(f"{len(CORPUS)} chunks indexed. Topics covered: tiers, storage, keys, regions, backups, roles, CLI, rate limits.")

## §2 — The retriever (offline TF-IDF, dense-swappable)

Identical interface to every retriever in this phase: `search(query, k)` returns
`(chunk_id, score, text)` triples, best first. We use TF-IDF so it runs with zero downloads; the
**swap point** to real dense embeddings is two lines, and nothing downstream changes.

The one number we care about today is that `score`. It is the pipeline's own estimate of *"how
related is my best chunk to this question?"* — and that estimate is exactly what the **relevance
gate** will threshold on to decide whether to answer at all.

In [ ]:
class Retriever:
    def __init__(self, ids, texts):
        self.ids, self.texts = ids, texts
        self.vec = TfidfVectorizer(stop_words="english")
        self.matrix = self.vec.fit_transform(texts)     # SWAP POINT: replace with dense embeddings

    def search(self, query, k=3):
        qv = self.vec.transform([query])
        sims = cosine_similarity(qv, self.matrix)[0]
        order = np.argsort(sims)[::-1][:k]
        return [(self.ids[i], float(sims[i]), self.texts[i]) for i in order]

retriever = Retriever(CHUNK_IDS, CHUNK_TEXT)

for q in ["How much storage does the Free tier include?", "What is Northwind's stock price?"]:
    print(f"Q: {q}")
    for cid, score, txt in retriever.search(q, k=2):
        print(f"   {cid}  score={score:.3f}  {txt[:60]}...")
    print()

Look at the two scores above. The in-scope question ("Free tier storage") pulls a chunk with a
**high** similarity; the out-of-scope one ("stock price") — a topic the corpus never mentions —
scrapes a **much lower** best score, because nothing in the docs shares its meaning. *That gap is
the signal abstention runs on.* Let's make it a rule.

## §3 — The problem: a naive generator never refuses

Before the fix, feel the failure. Here is a naive extractive "generator": it takes the single
top chunk and returns its first sentence as the answer — no gate, no provenance, no notion that the
context might be irrelevant. This is a fair caricature of an ungrounded LLM prompt: *"Here is some
context, answer the question,"* with nothing forcing it to check whether the context is on-topic.

In [ ]:
def first_sentence(text):
    return re.split(r'(?<=[.!?])\s+', text.strip())[0]

def naive_answer(query):
    top_id, score, top_text = retriever.search(query, k=1)[0]
    return first_sentence(top_text)          # always answers, from whatever ranked #1

for q in ["How do I rotate an API key?",          # in scope  -> fine
          "Who is the CEO of Northwind Cloud?"]:   # out of scope -> HALLUCINATION
    print(f"Q: {q}\nA: {naive_answer(q)}\n")

The second answer is the disaster in miniature. The corpus has **nothing** about a CEO, but the
retriever still had to return *something*, and the naive generator dutifully turned an irrelevant
chunk into a confident, false-sounding statement. No caveat, no citation, no way for the reader to
know it's baseless. In production this is how RAG systems quietly lie. Now we fix it.

## §4 — The grounded generator: gate → extract → cite → verify

We rebuild the generator around the three-part contract from §0. Read the four steps in the class
below — each maps to one production concept:

1. **Relevance gate** — if the best retrieval score is below `min_score`, the context is judged
   insufficient and we **abstain** immediately. (This is the anti-hallucination valve. In an LLM
   pipeline the same job is done by a grounding instruction *plus* this pre-filter — belt and suspenders.)
2. **Extract** — from the chunks that clear the gate, pick the sentence with the strongest overlap
   with the question. Offline this is token overlap; with an LLM it's the model composing an answer
   restricted to these sentences.
3. **Cite** — tag the chosen sentence with the `id` of the chunk it came from, e.g. `[D3]`. The
   citation is not decoration: it's the pointer a human or an eval follows to check the claim.
4. **Faithfulness self-check** — before returning, verify the answer sentence really is supported by
   its cited chunk. The naive way — "do they share words?" — is fooled by **ubiquitous terms**:
   *every* passage says "Northwind Cloud," so the brand name alone would fake support for the
   off-topic "who is the **CEO** of Northwind Cloud?" The fix is the same instinct that makes "the" a
   stopword: a term that appears in nearly every chunk (and nearly every question) carries no
   discriminating signal, so we add the **domain / product name to the stoplist** and count only the
   overlap that's left. If nothing *informative* ties the answer to the question, we abstain. (Word
   overlap is a deliberately simple stand-in; the production version of this check is an
   **LLM-as-judge faithfulness score**, which we build in L88.)

Two design choices worth naming, because they're what makes this robust:
- We extract across **every gated chunk**, not just the top-ranked one — the answer sentence can live in `hits[1]` even when `hits[0]` had the higher raw score.
- **Domain stopwords** (`northwind`, `cloud`) are dropped from the support count. You always know your corpus's subject, so its name is noise for *this* check — a standard, honest bit of tuning, not a special case.

In [ ]:
STOP = set('''a an the is are was were of to in on for and or with without your you it its this that
these those how what who when where which do does can could will would from at by as be been being
into per over under about not no yes if then than more most some any each only'''.split())

# Domain stopwords: the product name saturates the corpus AND the questions -> zero signal here.
DOMAIN_STOP = {"northwind", "cloud"}

def stem(w):
    # tiny suffix stripper so cost==costs, region==regions, limit==limits (a real retrieval fix)
    for suf in ("ing", "ed", "es", "s"):
        if w.endswith(suf) and len(w) - len(suf) >= 3:
            return w[: -len(suf)]
    return w

def content_words(text, extra_stop=frozenset()):
    return {stem(w) for w in re.findall(r"[a-z0-9]+", text.lower())
            if w not in STOP and w not in extra_stop and len(w) > 2}

def best_sentence_across(query, hits):
    """Best (support, sentence, chunk_id) over ALL retrieved chunks, ignoring domain stopwords.
    Always returns a sentence (falls back to the top chunk's first sentence) so the ABSTAIN
    decision lives entirely in the thresholds, not in whether a sentence was found."""
    qw = content_words(query, DOMAIN_STOP)
    fallback = re.split(r'(?<=[.!?])\s+', hits[0][2].strip())[0]
    best = (0, fallback, hits[0][0])
    for cid, _score, text in hits:
        for s in re.split(r'(?<=[.!?])\s+', text.strip()):
            support = len(qw & content_words(s, DOMAIN_STOP))
            if support > best[0]:
                best = (support, s, cid)
    return best                                          # (support, sentence, chunk_id)

class GroundedRAG:
    def __init__(self, retriever, min_score=0.10, min_support=1):
        self.retriever   = retriever      # relevance gate threshold
        self.min_score   = min_score      # below this best retrieval score => abstain (pre-filter)
        self.min_support = min_support    # answer must share >= this many INFORMATIVE words

    ABSTAIN = "I don't have information on that in the provided documents."

    def answer(self, query, k=3):
        hits = self.retriever.search(query, k=k)
        best_score = hits[0][1]

        # 1. RELEVANCE GATE (pre-filter) ----------------------------------
        if best_score < self.min_score:
            return {"answer": self.ABSTAIN, "citations": [], "abstained": True,
                    "reason": f"best retrieval score {best_score:.3f} < gate {self.min_score}"}

        # 2. EXTRACT best sentence across ALL gated chunks ----------------
        support, sent, cid = best_sentence_across(query, hits)

        # 4. FAITHFULNESS SELF-CHECK (informative overlap) ----------------
        if support < self.min_support:
            return {"answer": self.ABSTAIN, "citations": [], "abstained": True,
                    "reason": f"context on-topic but no INFORMATIVE word ties it to the question (support={support})"}

        # 3. CITE ----------------------------------------------------------
        return {"answer": f"{sent} [{cid}]", "citations": [cid], "abstained": False,
                "reason": f"grounded in {cid} (informative support={support})"}

grag = GroundedRAG(retriever)
print("GroundedRAG ready — gate + cross-chunk extract + informative-support check + cite.")

## §5 — Naive vs grounded, side by side

Same three questions through both generators. Watch the out-of-scope question flip from a confident
lie to an honest refusal — while the in-scope questions still get answered, now **with a citation you
can click back to**.

In [ ]:
demo_qs = [
    "How do I rotate an API key?",           # in scope
    "How much does the Pro tier cost?",       # in scope
    "Who is the CEO of Northwind Cloud?",     # OUT of scope -> must abstain
]
for q in demo_qs:
    n = naive_answer(q)
    g = grag.answer(q)
    tag = "ABSTAIN" if g["abstained"] else "ANSWER "
    print(f"Q: {q}")
    print(f"  naive   : {n}")
    print(f"  grounded[{tag}]: {g['answer']}")
    print(f"            why: {g['reason']}\n")

# 💡 EXPERIMENT: add "Does Northwind integrate with Salesforce?" — another topic the docs never
#    cover. Confirm grounded abstains while naive invents an answer. Then lower min_score to 0.0
#    and watch grounded start hallucinating too: the gate is doing real work.

## §6 — Measure it: you cannot trust what you don't score

"It abstains sometimes" is a vibe. Production needs numbers. We build a labeled set with **both**
kinds of question — answerable (a right chunk exists) and unanswerable (no chunk covers it) — and
score four things that together define a trustworthy RAG generator:

- **Answer accuracy** (answerable qs): did it produce the right fact, cited?
- **Citation correctness** (answered qs): does the cited chunk actually contain the answer?
- **Abstention recall** (unanswerable qs): of the questions it *should* refuse, how many did it?
- **Hallucination rate** — the headline safety number: fraction of unanswerable questions it
  **answered anyway**. Lower is better; `0.0` is the goal.

In [ ]:
# (question, gold_chunk_id or None if unanswerable, expected_keyword or None)
EVAL = [
    ("How much storage is in the Free tier?",        "D1", "5 gb"),
    ("What does the Pro tier cost?",                 "D2", "25"),
    ("How do I rotate an API key?",                  "D3", "rotate"),
    ("Which regions are available?",                 "D4", "region"),
    ("How often do backups run on Pro?",             "D5", "6 hours"),
    ("Who can delete a project?",                     "D6", "owner"),
    ("How do I install the CLI?",                     "D7", "npm"),
    ("What is the Pro tier rate limit?",             "D8", "1000"),
    # ---- unanswerable: nothing in the corpus covers these ----
    ("What is Northwind's stock price?",             None, None),
    ("Who is the CEO of Northwind Cloud?",           None, None),
    ("Does Northwind integrate with Salesforce?",    None, None),
    ("What is the company's phone number?",          None, None),
]

def evaluate(rag):
    ans_total = ans_ok = cite_total = cite_ok = 0
    unans_total = abstain_ok = 0
    for q, gold, kw in EVAL:
        r = rag.answer(q)
        if gold is not None:                           # answerable
            ans_total += 1
            text = r["answer"].lower()
            if not r["abstained"] and kw in text:
                ans_ok += 1
            if not r["abstained"]:                     # citation correctness
                cite_total += 1
                if gold in r["citations"]:
                    cite_ok += 1
        else:                                          # unanswerable
            unans_total += 1
            if r["abstained"]:
                abstain_ok += 1
    return {
        "answer_accuracy":   ans_ok    / ans_total,
        "citation_correct":  cite_ok   / max(cite_total, 1),
        "abstention_recall": abstain_ok/ unans_total,
        "hallucination_rate": (unans_total - abstain_ok) / unans_total,
    }

def show(name, m):
    print(f"{name:>16}:  answer_acc={m['answer_accuracy']:.2f}  "
          f"cite_correct={m['citation_correct']:.2f}  "
          f"abstain_recall={m['abstention_recall']:.2f}  "
          f"HALLUCINATION={m['hallucination_rate']:.2f}")

show("grounded", evaluate(grag))

### The controlled experiment that proves the gate earns its keep

To show the numbers aren't a coincidence, run the *same* eval against a version of the pipeline with
the relevance gate switched off (`min_score=0.0`, `min_support=0`) — i.e. forced to always answer.
This isolates exactly what abstention buys you.

In [ ]:
always_answer = GroundedRAG(retriever, min_score=0.0, min_support=0)  # gate disabled

show("gate OFF", evaluate(always_answer))
show("gate ON ", evaluate(grag))

print()
off = evaluate(always_answer)["hallucination_rate"]
on  = evaluate(grag)["hallucination_rate"]
print(f"Turning the gate on cut the hallucination rate from {off:.2f} to {on:.2f} "
      f"on the unanswerable questions — at no cost to answer accuracy on the answerable ones.")

# 💡 EXPERIMENT: sweep min_score from 0.0 to 0.30 and print hallucination_rate vs answer_accuracy at
#    each step. You'll see the classic tradeoff curve: too high a gate and you start refusing good
#    questions (answer_accuracy drops); too low and hallucinations creep back. This is the
#    precision/recall dial of abstention — L88 turns it into a proper metric.

## §7 — The same contract with a real LLM (Claude)

Offline we *enforced* grounding by construction (extract-only). With a real generative model you get
fluent, synthesized answers — but now grounding is a thing you must **instruct and then verify**,
because the model *can* pull from its own training memory. Three levers do it, and they map 1:1 onto
what we built offline:

- **A grounding system prompt** — "answer *only* from the context; if it isn't there, say you don't know" (offline: extract-only + gate).
- **Inline citations** — require `[D#]` tags so every claim is traceable (offline: the `[cid]` tag).
- **A pre-filter gate** — still abstain *before* calling the model when retrieval is weak, to save a call and remove the temptation entirely (offline: `min_score`).

The cell is guarded by `HAS_KEY`, so it runs live only when you provide a key and otherwise prints
the prompt design and skips — the notebook stays green offline.

In [ ]:
GROUNDING_SYSTEM = """You answer questions using ONLY the numbered context passages provided.
Rules:
1. Use only facts stated in the context. Never use outside knowledge.
2. After every sentence, cite the passage id it came from, like [D3].
3. If the context does not contain the answer, reply exactly:
   "I don't have information on that in the provided documents."
   Do not guess, and do not apologize at length."""

def format_context(hits):
    return "\n".join(f"[{cid}] {txt}" for cid, _score, txt in hits)

def claude_grounded_answer(query, k=3, min_score=0.10):
    hits = retriever.search(query, k=k)
    if hits[0][1] < min_score:                     # gate BEFORE spending a call
        return "I don't have information on that in the provided documents. (gate: weak retrieval)"
    import anthropic
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=300,
        system=GROUNDING_SYSTEM,
        messages=[{"role": "user",
                   "content": f"Context:\n{format_context(hits)}\n\nQuestion: {query}"}],
    )
    return msg.content[0].text

if HAS_KEY:
    for q in ["How do I rotate an API key?", "Who is the CEO of Northwind Cloud?"]:
        print(f"Q: {q}\nA: {claude_grounded_answer(q)}\n")
else:
    print("No API key -> skipping live Claude call (offline run stays green).")
    print("\nThe grounding system prompt that does the work:\n")
    print(GROUNDING_SYSTEM)

# 💡 EXPERIMENT (needs a key): delete rule #3 from GROUNDING_SYSTEM and re-ask the CEO question.
#    Watch a fluent model invent a plausible name once you remove the abstention instruction —
#    the same failure the offline gate prevented, now in a much more convincing voice.

## §8 — Recap & what you built

You turned a RAG pipeline that *always answers* into one that **answers only from evidence, cites
where each claim came from, and refuses when the evidence isn't there** — and you proved it with a
hallucination-rate number, not a vibe.

**The mental model to keep:**

> A RAG generator's ground truth is its retrieved context, not the world. Faithfulness = every claim
> traces to a chunk. The relevance gate is the valve that stops the system answering when there's
> nothing to be faithful *to*.

**The three habits, and their production forms:**

- **Grounding** — offline: extract-only. Production: a strict system prompt + you still verify.
- **Citations** — `[D#]` tags. They're not UI polish; they are what makes an answer *checkable*, and they feed the faithfulness eval in L88.
- **Abstention** — a relevance gate before generation. It's a precision/recall dial: tune it against your own hallucination-vs-coverage tradeoff.

**FM5 status:** addressed. The system now has provenance and a refusal path — the two things a naive
RAG lacks.

**Next — L88: RAG evaluation.** We've been eyeballing one keyword per question. L88 makes it rigorous:
retrieval metrics (Recall@k, MRR, nDCG — you met these in L86) *plus* generation metrics
(faithfulness, answer relevance, and an **LLM-as-judge** that scores whether each cited claim is
actually supported — the automated version of the self-check you hand-built in §4). That's the last
piece before the L89 capstone where we package the whole Phase 10 stack into a deployable service.

> **One-line takeaway:** *In production RAG, "I don't know" is a feature — and a citation is what
> turns an answer into something a human can trust.*